# Day 6：从零手写 RLHF-PPO 训练（本周最重要！）

> **目标**：从零实现完整的 RLHF-PPO 训练循环——`四模型构建 → Rollout 生成 → Reward 计算 → GAE 优势估计 → PPO 策略更新`，在 GPT-2 small 上完成 RLHF 训练，可视化训练过程（reward / KL / loss 曲线），对比 RLHF 前后的生成质量，并分析超参数（β / clip_ratio）的影响。
>
> RLHF-PPO 训练循环是**面试 Tier 1 必须能闭眼手写**的内容，也是本周全部理论的工程落地。

---

## 实现路线图

```
Part 1: RLHF-PPO 数学回顾
  四模型架构 + 三部分 Loss 速查

Part 2: 模型准备
  GPT-2 → Actor / Critic / Reference / Reward Model

Part 3: Rollout（生成 Response）
  Actor 自回归生成 → 收集 log_probs, values, response_ids

Part 4: Reward 计算
  RM 打分 + Reference KL → per-token reward 构建

Part 5: GAE 计算
  δ_t → A_t^GAE → returns（反向递推）

Part 6: PPO 更新
  ratio → clipped policy loss + value loss → backward

Part 7: 完整训练循环
  for iteration: rollout → reward → GAE → PPO update

Part 8: 训练运行与监控
  执行训练 + 训练曲线可视化

Part 9: 生成质量对比
  RLHF 前 vs 后的生成结果对比

Part 10: 超参数分析
  β / clip_ratio 对训练的影响

Part 11: 面试要点 + 总结
  核心代码速查 + Day 7 衔接
```

In [ ]:
import random
import copy
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import matplotlib.pyplot as plt
from collections import defaultdict

try:
    from transformers import GPT2LMHeadModel, GPT2Model, GPT2Tokenizer
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'transformers'])
    from transformers import GPT2LMHeadModel, GPT2Model, GPT2Tokenizer

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

torch.manual_seed(42)
np.random.seed(42)
random.seed(42)

## Part 1：RLHF-PPO 数学回顾

### 1.1 四模型架构

```
┌─────────────────┐              ┌─────────────────┐
│  Actor (π_θ)     │              │  Critic (V_ϕ)    │
│  ← 梯度更新       │              │  ← 梯度更新       │
└────────┬────────┘              └────────┬────────┘
         │                                │
┌────────┴────────┐              ┌────────┴────────┐
│  Reference (π_ref)│              │  Reward Model    │
│  ← 冻结           │              │  ← 冻结           │
└─────────────────┘              └─────────────────┘
```

### 1.2 三个核心公式

**Policy Loss (PPO-Clip)**：

$$L^{\text{policy}} = -\frac{1}{|\mathcal{M}|} \sum_{t \in \mathcal{M}} \min\left(r_t \hat{A}_t,\; \text{clip}(r_t, 1\pm\epsilon)\hat{A}_t\right)$$

**Value Loss**：

$$L^{\text{value}} = \frac{1}{|\mathcal{M}|} \sum_{t \in \mathcal{M}} (V_\phi(s_t) - G_t)^2$$

**Per-token Reward (KL Reward Shaping)**：

$$r_t = -\beta \cdot \text{kl}_t + \mathbb{1}_{t=T} \cdot r_\phi(x, y)$$

### 1.3 一轮迭代流程

```
Phase 1: Rollout → Phase 2: Scoring → Phase 3: GAE → Phase 4: PPO Update
```

## Part 2：模型准备

我们使用 GPT-2 small (124M) 构建 RLHF 的四个模型。在教学场景下，为了降低显存和加速训练，我们采用以下设计：

| 模型 | 实现方式 | 是否更新 |
|------|---------|--------|
| Actor | `GPT2LMHeadModel` | ✓ |
| Critic | `GPT2Model` + `ValueHead` | ✓ |
| Reference | `GPT2LMHeadModel`（Actor 的冻结副本） | ✗ |
| Reward Model | 简化版 RM（基于规则 + 小模型） | ✗ |

Critic 独立于 Actor（不共享 backbone），这样梯度不冲突，更易调试。

In [ ]:
# ========== Tokenizer ==========
tokenizer = GPT2Tokenizer.from_pretrained('gpt2')
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'left'

# ========== Actor ==========
actor = GPT2LMHeadModel.from_pretrained('gpt2').to(device)
actor.config.pad_token_id = tokenizer.pad_token_id

# ========== Reference (Actor 的冻结副本) ==========
reference = GPT2LMHeadModel.from_pretrained('gpt2').to(device)
reference.config.pad_token_id = tokenizer.pad_token_id
for param in reference.parameters():
    param.requires_grad = False
reference.eval()

# ========== Critic ==========
class ValueHead(nn.Module):
    """Critic: GPT-2 backbone + linear value head → 每个 token 位置输出 V(s_t)。"""
    def __init__(self, model_name='gpt2'):
        super().__init__()
        self.backbone = GPT2Model.from_pretrained(model_name)
        hidden_size = self.backbone.config.hidden_size
        self.value_head = nn.Linear(hidden_size, 1, bias=False)
    
    def forward(self, input_ids, attention_mask=None):
        outputs = self.backbone(input_ids=input_ids, attention_mask=attention_mask)
        hidden = outputs.last_hidden_state          # (batch, seq, hidden)
        values = self.value_head(hidden).squeeze(-1) # (batch, seq)
        return values

critic = ValueHead('gpt2').to(device)

# ========== Reward Model (简化版) ==========
class SimpleRewardModel(nn.Module):
    """
    简化版 RM：GPT-2 backbone + reward head。
    在教学场景下，我们结合模型打分与基于规则的启发式来产生奖励信号，
    模拟一个训练好的 RM 的行为。
    """
    def __init__(self, model_name='gpt2'):
        super().__init__()
        self.backbone = GPT2Model.from_pretrained(model_name)
        hidden_size = self.backbone.config.hidden_size
        self.reward_head = nn.Linear(hidden_size, 1, bias=False)
    
    def forward(self, input_ids, attention_mask=None):
        outputs = self.backbone(input_ids=input_ids, attention_mask=attention_mask)
        hidden = outputs.last_hidden_state
        if attention_mask is not None:
            seq_lengths = attention_mask.sum(dim=1) - 1
        else:
            seq_lengths = torch.full((input_ids.size(0),), input_ids.size(1) - 1,
                                     device=input_ids.device)
        batch_idx = torch.arange(hidden.size(0), device=hidden.device)
        last_hidden = hidden[batch_idx, seq_lengths]
        reward = self.reward_head(last_hidden).squeeze(-1)
        return reward

reward_model = SimpleRewardModel('gpt2').to(device)
for param in reward_model.parameters():
    param.requires_grad = False
reward_model.eval()

print("四模型构建完成 ✓")
print(f"  Actor params:     {sum(p.numel() for p in actor.parameters()):,}")
print(f"  Critic params:    {sum(p.numel() for p in critic.parameters()):,}")
print(f"  Reference params: {sum(p.numel() for p in reference.parameters()):,} (frozen)")
print(f"  RM params:        {sum(p.numel() for p in reward_model.parameters()):,} (frozen)")

In [ ]:
# ========== Prompt 数据集 ==========
PROMPTS = [
    "Explain what machine learning is in simple terms.",
    "What is the capital of France and why is it famous?",
    "How does photosynthesis work?",
    "Describe the process of making bread.",
    "What are the benefits of regular exercise?",
    "Explain how the internet works.",
    "What is climate change and why does it matter?",
    "How do computers store information?",
    "What is the theory of evolution?",
    "Explain what a neural network is.",
    "How does gravity work?",
    "What are the main causes of pollution?",
    "Describe how vaccines protect against diseases.",
    "What is renewable energy?",
    "How do airplanes fly?",
    "What is quantum computing?",
]


def get_prompt_batch(prompts, tokenizer, batch_size, max_length=32):
    """随机采样一个 batch 的 prompts 并 tokenize。"""
    selected = random.choices(prompts, k=batch_size)
    encoding = tokenizer(
        selected,
        max_length=max_length,
        padding='max_length',
        truncation=True,
        return_tensors='pt'
    )
    return encoding['input_ids'].to(device), encoding['attention_mask'].to(device), selected


# 测试
test_ids, test_mask, test_texts = get_prompt_batch(PROMPTS, tokenizer, batch_size=2)
print(f"Prompt batch shape: {test_ids.shape}")
print(f"Sample prompt: {test_texts[0]}")

## Part 3：Rollout（生成 Response）

Rollout 是 RLHF 的 Phase 1：Actor 自回归生成 response，同时收集训练所需的数据。

```
输入: prompt_ids (batch, prompt_len)

for t = 1, 2, ..., max_new_tokens:
    logits_t = Actor(input_ids)[:, -1, :]       ← 最后一个位置的 logits
    y_t ~ Categorical(softmax(logits_t))         ← 采样下一个 token
    log_prob_t = log π_θ(y_t | s_t)              ← 记录 log prob
    value_t = V_ϕ(s_t)                           ← Critic 估值

输出:
    response_ids: (batch, response_len)          ← 生成的 token 序列
    old_log_probs: (batch, response_len)         ← 每个 token 的 log π_old
    old_values: (batch, response_len)            ← 每个 token 的 V_old
    full_ids: (batch, prompt_len + response_len) ← 完整序列
```

所有操作在 `torch.no_grad()` 下执行——这是推理阶段，不需要梯度。

In [ ]:
@torch.no_grad()
def generate_rollout(actor, critic, prompt_ids, prompt_mask,
                     max_new_tokens=48, temperature=1.0):
    """
    RLHF Rollout: Actor 自回归生成 response，同时收集 log_probs 和 values。
    
    返回:
        full_ids:      (batch, prompt_len + response_len) 完整 token 序列
        response_ids:  (batch, response_len) 生成的 response
        old_log_probs: (batch, response_len) rollout 时的 log π_old
        old_values:    (batch, response_len) rollout 时的 V_old
        response_mask: (batch, response_len) response 有效区域掩码
    """
    actor.eval()
    critic.eval()
    
    batch_size = prompt_ids.size(0)
    prompt_len = prompt_ids.size(1)
    
    generated = prompt_ids.clone()                 # (batch, prompt_len)
    gen_mask = prompt_mask.clone()
    all_log_probs = []
    all_values = []
    all_tokens = []
    finished = torch.zeros(batch_size, dtype=torch.bool, device=device)
    
    for t in range(max_new_tokens):
        outputs = actor(input_ids=generated, attention_mask=gen_mask)
        logits = outputs.logits[:, -1, :] / temperature  # (batch, vocab)
        
        values = critic(input_ids=generated, attention_mask=gen_mask)
        value_t = values[:, -1]                          # (batch,)
        
        probs = F.softmax(logits, dim=-1)
        next_token = torch.multinomial(probs, num_samples=1)  # (batch, 1)
        
        log_prob = F.log_softmax(logits, dim=-1)
        token_log_prob = log_prob.gather(1, next_token).squeeze(1)  # (batch,)
        
        # 已结束的序列不再产生有效 token
        token_log_prob = token_log_prob.masked_fill(finished, 0.0)
        value_t = value_t.masked_fill(finished, 0.0)
        
        all_log_probs.append(token_log_prob)
        all_values.append(value_t)
        all_tokens.append(next_token.squeeze(1))
        
        generated = torch.cat([generated, next_token], dim=1)
        gen_mask = torch.cat([gen_mask, (~finished).long().unsqueeze(1)], dim=1)
        
        finished = finished | (next_token.squeeze(1) == tokenizer.eos_token_id)
        if finished.all():
            break
    
    response_ids = torch.stack(all_tokens, dim=1)       # (batch, response_len)
    old_log_probs = torch.stack(all_log_probs, dim=1)   # (batch, response_len)
    old_values = torch.stack(all_values, dim=1)         # (batch, response_len)
    
    # response mask: 从第一个 token 到 EOS（含）为 1，之后为 0
    response_mask = torch.ones_like(response_ids, dtype=torch.float)
    for i in range(batch_size):
        eos_positions = (response_ids[i] == tokenizer.eos_token_id).nonzero(as_tuple=False)
        if len(eos_positions) > 0:
            first_eos = eos_positions[0].item()
            response_mask[i, first_eos + 1:] = 0.0
    
    full_ids = generated  # (batch, prompt_len + response_len)
    
    return {
        'full_ids': full_ids,
        'response_ids': response_ids,
        'old_log_probs': old_log_probs,
        'old_values': old_values,
        'response_mask': response_mask,
        'prompt_len': prompt_len,
    }


# 测试 rollout
test_rollout = generate_rollout(actor, critic, test_ids, test_mask, max_new_tokens=20)
print(f"Full sequence shape: {test_rollout['full_ids'].shape}")
print(f"Response shape:      {test_rollout['response_ids'].shape}")
print(f"Log probs shape:     {test_rollout['old_log_probs'].shape}")
print(f"Values shape:        {test_rollout['old_values'].shape}")
print(f"\nGenerated text:")
print(tokenizer.decode(test_rollout['full_ids'][0], skip_special_tokens=True)[:200])
print("\nRollout 定义完成 ✓")

## Part 4：Reward 计算

Phase 2: 使用 RM 和 Reference 构建 per-token reward。

$$r_t = -\beta \cdot \text{kl}_t + \mathbb{1}_{t=T} \cdot r_\phi(x, y)$$

其中 $\text{kl}_t = \log \pi_\theta(y_t \mid s_t) - \log \pi_{\text{ref}}(y_t \mid s_t)$

```
Step 1: RM 对完整序列打分 → rm_score (scalar per sample)
Step 2: Reference 计算 ref_log_probs (per token)
Step 3: kl_t = old_log_probs_t - ref_log_probs_t
Step 4: rewards_t = -β * kl_t
Step 5: rewards[last_token] += rm_score
```

In [ ]:
@torch.no_grad()
def compute_rewards(reward_model, reference, rollout_data,
                    prompt_ids, prompt_mask, kl_coef=0.1):
    """
    计算 per-token reward = KL penalty + 终末 RM score。
    
    返回:
        rewards:       (batch, response_len) per-token reward
        rm_scores:     (batch,) RM 打分
        kl_per_token:  (batch, response_len) per-token KL
        mean_kl:       scalar, batch 平均 KL
    """
    full_ids = rollout_data['full_ids']
    response_ids = rollout_data['response_ids']
    old_log_probs = rollout_data['old_log_probs']
    response_mask = rollout_data['response_mask']
    prompt_len = rollout_data['prompt_len']
    batch_size, response_len = response_ids.shape
    
    # --- Step 1: RM 打分 ---
    full_mask = torch.ones_like(full_ids, dtype=torch.long)
    rm_scores = reward_model(full_ids, full_mask)  # (batch,)
    
    # --- Step 2: Reference log probs ---
    ref_outputs = reference(input_ids=full_ids, attention_mask=full_mask)
    ref_logits = ref_outputs.logits[:, prompt_len - 1:-1, :]  # 对齐 response 位置
    ref_log_probs_full = F.log_softmax(ref_logits, dim=-1)
    
    # 只取 response_len 长度（可能因 EOS 提前停止而更短）
    actual_len = min(ref_log_probs_full.size(1), response_len)
    ref_log_probs = torch.zeros(batch_size, response_len, device=device)
    for t in range(actual_len):
        ref_log_probs[:, t] = ref_log_probs_full[:, t, :].gather(
            1, response_ids[:, t:t+1]
        ).squeeze(1)
    
    # --- Step 3-5: 构建 per-token reward ---
    kl_per_token = old_log_probs - ref_log_probs   # (batch, response_len)
    rewards = -kl_coef * kl_per_token               # KL penalty
    
    # 在每个序列最后一个有效 token 上加 RM score
    for i in range(batch_size):
        valid_len = int(response_mask[i].sum().item())
        if valid_len > 0:
            rewards[i, valid_len - 1] += rm_scores[i]
    
    rewards = rewards * response_mask  # 无效位置清零
    
    mean_kl = (kl_per_token * response_mask).sum() / response_mask.sum()
    
    return {
        'rewards': rewards,
        'rm_scores': rm_scores,
        'kl_per_token': kl_per_token,
        'mean_kl': mean_kl.item(),
    }


# 测试
test_rewards = compute_rewards(reward_model, reference, test_rollout,
                                test_ids, test_mask, kl_coef=0.1)
print(f"Rewards shape:  {test_rewards['rewards'].shape}")
print(f"RM scores:      {test_rewards['rm_scores'].tolist()}")
print(f"Mean KL:        {test_rewards['mean_kl']:.4f}")
print(f"Rewards sample: {test_rewards['rewards'][0, :10].tolist()}")
print("\nReward 计算定义完成 ✓")

## Part 5：GAE 计算

Phase 3: 用 per-token reward 和 Critic values 计算 Advantage 和 Returns。

$$\delta_t = r_t + \gamma \cdot V(s_{t+1}) - V(s_t)$$

$$\hat{A}_t = \sum_{l=0}^{T-t-1} (\gamma\lambda)^l \delta_{t+l} = \delta_t + \gamma\lambda \cdot \hat{A}_{t+1}$$

$$G_t = \hat{A}_t + V(s_t)$$

RLHF 中通常 $\gamma = 1.0$（response 较短，不需要折扣）。

In [ ]:
def compute_gae(rewards, values, response_mask, gamma=1.0, lam=0.95):
    """
    GAE 计算（反向递推）。
    
    rewards:       (batch, seq_len)
    values:        (batch, seq_len)
    response_mask: (batch, seq_len)
    
    返回: advantages (batch, seq_len), returns (batch, seq_len)
    """
    batch_size, seq_len = rewards.shape
    advantages = torch.zeros_like(rewards)
    last_gae = torch.zeros(batch_size, device=rewards.device)
    
    for t in reversed(range(seq_len)):
        if t == seq_len - 1:
            next_value = 0.0
        else:
            next_value = values[:, t + 1]
        
        # 如果 t+1 位置已经超出 mask，next_value 置 0
        if t < seq_len - 1:
            next_mask = response_mask[:, t + 1]
            next_value = next_value * next_mask
        
        delta = rewards[:, t] + gamma * next_value - values[:, t]
        last_gae = delta + gamma * lam * last_gae * response_mask[:, t]
        advantages[:, t] = last_gae
    
    advantages = advantages * response_mask
    returns = advantages + values
    
    return advantages, returns


# 测试
test_adv, test_ret = compute_gae(
    test_rewards['rewards'],
    test_rollout['old_values'],
    test_rollout['response_mask']
)
print(f"Advantages shape: {test_adv.shape}")
print(f"Returns shape:    {test_ret.shape}")
print(f"Advantages (sample): {test_adv[0, :5].tolist()}")
print(f"Returns (sample):    {test_ret[0, :5].tolist()}")
print("\nGAE 计算定义完成 ✓")

## Part 6：PPO 更新

Phase 4: 用收集到的数据更新 Actor 和 Critic。

关键步骤：
1. **重新前向传播** Actor 和 Critic，得到 `new_log_probs` 和 `new_values`
2. 计算 `ratio = exp(new_log_probs - old_log_probs)`
3. **Policy Loss**: PPO-Clip
4. **Value Loss**: MSE(new_values, returns)
5. 反向传播 + 梯度裁剪 + optimizer step

注意：这里需要**梯度**——这是整个 RLHF 流程中唯一的训练阶段。

In [ ]:
def get_log_probs_and_values(actor, critic, full_ids, response_ids,
                              prompt_len, response_mask):
    """
    重新前向传播 Actor 和 Critic，计算当前的 log_probs 和 values。
    这一步需要梯度，用于 PPO 更新。
    """
    batch_size, response_len = response_ids.shape
    full_mask = torch.ones_like(full_ids, dtype=torch.long)
    
    # Actor: 重新计算 log probs
    actor_outputs = actor(input_ids=full_ids, attention_mask=full_mask)
    logits = actor_outputs.logits[:, prompt_len - 1:-1, :]  # 对齐 response
    actual_len = min(logits.size(1), response_len)
    
    new_log_probs = torch.zeros(batch_size, response_len, device=device)
    log_probs_all = F.log_softmax(logits, dim=-1)
    for t in range(actual_len):
        new_log_probs[:, t] = log_probs_all[:, t, :].gather(
            1, response_ids[:, t:t+1]
        ).squeeze(1)
    
    # Critic: 重新计算 values
    all_values = critic(input_ids=full_ids, attention_mask=full_mask)
    new_values = all_values[:, prompt_len:prompt_len + response_len]
    
    return new_log_probs, new_values


def ppo_update(actor, critic, optimizer_actor, optimizer_critic,
               rollout_data, advantages, returns,
               clip_ratio=0.2, value_coef=0.5, max_grad_norm=0.5,
               ppo_epochs=2):
    """
    PPO 更新：多个 epoch 更新 Actor 和 Critic。
    
    返回: dict of metrics (policy_loss, value_loss, clip_fraction, approx_kl)
    """
    full_ids = rollout_data['full_ids']
    response_ids = rollout_data['response_ids']
    old_log_probs = rollout_data['old_log_probs']
    response_mask = rollout_data['response_mask']
    prompt_len = rollout_data['prompt_len']
    
    # 标准化 advantages
    valid_advs = advantages[response_mask.bool()]
    if valid_advs.numel() > 1:
        adv_mean = valid_advs.mean()
        adv_std = valid_advs.std() + 1e-8
        advantages = (advantages - adv_mean) / adv_std * response_mask
    
    total_metrics = defaultdict(float)
    
    actor.train()
    critic.train()
    
    for epoch in range(ppo_epochs):
        new_log_probs, new_values = get_log_probs_and_values(
            actor, critic, full_ids, response_ids, prompt_len, response_mask
        )
        
        # ===== Policy Loss (PPO-Clip) =====
        log_ratio = new_log_probs - old_log_probs
        ratio = torch.exp(log_ratio)
        
        surr1 = ratio * advantages
        surr2 = torch.clamp(ratio, 1.0 - clip_ratio, 1.0 + clip_ratio) * advantages
        policy_loss = -torch.min(surr1, surr2)
        policy_loss = (policy_loss * response_mask).sum() / response_mask.sum()
        
        # ===== Value Loss (MSE) =====
        value_loss = (new_values - returns.detach()) ** 2
        value_loss = (value_loss * response_mask).sum() / response_mask.sum()
        
        # ===== Total Loss =====
        total_loss = policy_loss + value_coef * value_loss
        
        # ===== Backward =====
        optimizer_actor.zero_grad()
        optimizer_critic.zero_grad()
        total_loss.backward()
        torch.nn.utils.clip_grad_norm_(actor.parameters(), max_grad_norm)
        torch.nn.utils.clip_grad_norm_(critic.parameters(), max_grad_norm)
        optimizer_actor.step()
        optimizer_critic.step()
        
        # ===== Metrics =====
        with torch.no_grad():
            clip_fraction = ((ratio - 1.0).abs() > clip_ratio).float()
            clip_fraction = (clip_fraction * response_mask).sum() / response_mask.sum()
            approx_kl = ((ratio - 1.0) - log_ratio)
            approx_kl = (approx_kl * response_mask).sum() / response_mask.sum()
        
        total_metrics['policy_loss'] += policy_loss.item()
        total_metrics['value_loss'] += value_loss.item()
        total_metrics['clip_fraction'] += clip_fraction.item()
        total_metrics['approx_kl'] += approx_kl.item()
    
    for k in total_metrics:
        total_metrics[k] /= ppo_epochs
    
    return dict(total_metrics)


print("PPO 更新函数定义完成 ✓")

## Part 7：完整训练循环

将 Phase 1-4 串联为完整的 RLHF-PPO 训练循环：

```
for iteration = 1 to N:
    1. 采样 prompts
    2. Rollout:  Actor 生成 response（收集 old_log_probs, old_values）
    3. Scoring:  RM 打分 + Reference KL → per-token reward
    4. GAE:      advantages, returns
    5. PPO:      更新 Actor + Critic（K epochs）
    6. Log:      记录 metrics
```

In [ ]:
def rlhf_train(actor, critic, reference, reward_model, tokenizer, prompts,
               n_iterations=50, batch_size=4, max_new_tokens=48,
               lr_actor=1e-5, lr_critic=5e-5, kl_coef=0.1,
               clip_ratio=0.2, value_coef=0.5, gamma=1.0, lam=0.95,
               ppo_epochs=2, max_grad_norm=0.5, prompt_max_len=32,
               log_interval=5):
    """
    RLHF-PPO 完整训练循环。
    
    这个函数对应 Day 5 第八节的伪代码：
    Phase 1: Rollout → Phase 2: Scoring → Phase 3: GAE → Phase 4: PPO Update
    """
    optimizer_actor = optim.AdamW(actor.parameters(), lr=lr_actor, weight_decay=0.01)
    optimizer_critic = optim.AdamW(critic.parameters(), lr=lr_critic, weight_decay=0.01)
    
    history = defaultdict(list)
    
    for iteration in range(1, n_iterations + 1):
        # ===== Step 1: 采样 Prompts =====
        prompt_ids, prompt_mask, prompt_texts = get_prompt_batch(
            prompts, tokenizer, batch_size, max_length=prompt_max_len
        )
        
        # ===== Phase 1: Rollout (no grad) =====
        rollout_data = generate_rollout(
            actor, critic, prompt_ids, prompt_mask,
            max_new_tokens=max_new_tokens
        )
        
        # ===== Phase 2: Scoring (no grad) =====
        reward_data = compute_rewards(
            reward_model, reference, rollout_data,
            prompt_ids, prompt_mask, kl_coef=kl_coef
        )
        
        # ===== Phase 3: GAE =====
        advantages, returns = compute_gae(
            reward_data['rewards'],
            rollout_data['old_values'],
            rollout_data['response_mask'],
            gamma=gamma, lam=lam
        )
        
        # ===== Phase 4: PPO Update =====
        ppo_metrics = ppo_update(
            actor, critic, optimizer_actor, optimizer_critic,
            rollout_data, advantages, returns,
            clip_ratio=clip_ratio, value_coef=value_coef,
            max_grad_norm=max_grad_norm, ppo_epochs=ppo_epochs
        )
        
        # ===== 记录 Metrics =====
        history['mean_reward'].append(reward_data['rm_scores'].mean().item())
        history['mean_kl'].append(reward_data['mean_kl'])
        history['policy_loss'].append(ppo_metrics['policy_loss'])
        history['value_loss'].append(ppo_metrics['value_loss'])
        history['clip_fraction'].append(ppo_metrics['clip_fraction'])
        history['approx_kl'].append(ppo_metrics['approx_kl'])
        
        mean_response_len = rollout_data['response_mask'].sum(1).float().mean().item()
        history['response_len'].append(mean_response_len)
        
        # ===== 日志 =====
        if iteration % log_interval == 0 or iteration == 1:
            print(f"Iter {iteration:3d}/{n_iterations} | "
                  f"Reward: {history['mean_reward'][-1]:+.3f} | "
                  f"KL: {history['mean_kl'][-1]:.4f} | "
                  f"PL: {ppo_metrics['policy_loss']:.4f} | "
                  f"VL: {ppo_metrics['value_loss']:.4f} | "
                  f"Clip: {ppo_metrics['clip_fraction']:.2%} | "
                  f"Len: {mean_response_len:.1f}")
    
    return dict(history)


print("RLHF 训练循环定义完成 ✓")

## Part 8：训练运行与监控

### 8.1 超参数设置

| 超参数 | 值 | 说明 |
|--------|------|------|
| `n_iterations` | 50 | 训练轮数（教学环境） |
| `batch_size` | 4 | Prompt batch 大小 |
| `max_new_tokens` | 48 | 最大生成长度 |
| `lr_actor` | 1e-5 | Actor 学习率 |
| `lr_critic` | 5e-5 | Critic 学习率（通常更大） |
| `kl_coef` (β) | 0.1 | KL penalty 系数 |
| `clip_ratio` (ε) | 0.2 | PPO clip 范围 |
| `ppo_epochs` (K) | 2 | 每轮 rollout 的 PPO 更新次数 |
| `gamma` | 1.0 | 折扣因子（不折扣） |
| `lam` (λ) | 0.95 | GAE 参数 |

In [ ]:
# 保存 RLHF 前的 Actor 权重，用于后续对比
actor_before_state = copy.deepcopy(actor.state_dict())

print("=" * 70)
print("开始 RLHF-PPO 训练")
print("=" * 70)

history = rlhf_train(
    actor=actor,
    critic=critic,
    reference=reference,
    reward_model=reward_model,
    tokenizer=tokenizer,
    prompts=PROMPTS,
    n_iterations=50,
    batch_size=4,
    max_new_tokens=48,
    lr_actor=1e-5,
    lr_critic=5e-5,
    kl_coef=0.1,
    clip_ratio=0.2,
    ppo_epochs=2,
    log_interval=5,
)

print("\n" + "=" * 70)
print("RLHF-PPO 训练完成!")
print("=" * 70)

In [ ]:
# ========== 训练曲线可视化 ==========
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

# 1) Mean Reward
axes[0, 0].plot(history['mean_reward'], 'b-', linewidth=1.5)
axes[0, 0].set_title('Mean RM Reward')
axes[0, 0].set_xlabel('Iteration')
axes[0, 0].set_ylabel('Reward')
axes[0, 0].grid(True, alpha=0.3)

# 2) Mean KL
axes[0, 1].plot(history['mean_kl'], 'r-', linewidth=1.5)
axes[0, 1].set_title('Mean KL Divergence')
axes[0, 1].set_xlabel('Iteration')
axes[0, 1].set_ylabel('KL')
axes[0, 1].grid(True, alpha=0.3)

# 3) Policy Loss
axes[0, 2].plot(history['policy_loss'], 'g-', linewidth=1.5)
axes[0, 2].set_title('Policy Loss')
axes[0, 2].set_xlabel('Iteration')
axes[0, 2].set_ylabel('Loss')
axes[0, 2].grid(True, alpha=0.3)

# 4) Value Loss
axes[1, 0].plot(history['value_loss'], 'm-', linewidth=1.5)
axes[1, 0].set_title('Value Loss')
axes[1, 0].set_xlabel('Iteration')
axes[1, 0].set_ylabel('Loss')
axes[1, 0].grid(True, alpha=0.3)

# 5) Clip Fraction
axes[1, 1].plot(history['clip_fraction'], 'c-', linewidth=1.5)
axes[1, 1].set_title('Clip Fraction')
axes[1, 1].set_xlabel('Iteration')
axes[1, 1].set_ylabel('Fraction')
axes[1, 1].axhline(y=0.1, color='gray', linestyle='--', alpha=0.5, label='Healthy (~0.1)')
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3)

# 6) Response Length
axes[1, 2].plot(history['response_len'], 'orange', linewidth=1.5)
axes[1, 2].set_title('Mean Response Length')
axes[1, 2].set_xlabel('Iteration')
axes[1, 2].set_ylabel('Tokens')
axes[1, 2].grid(True, alpha=0.3)

plt.suptitle('RLHF-PPO Training Metrics', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print(f"\n训练摘要:")
print(f"  初始 Reward: {history['mean_reward'][0]:+.4f} → 最终: {history['mean_reward'][-1]:+.4f}")
print(f"  初始 KL:     {history['mean_kl'][0]:.4f} → 最终: {history['mean_kl'][-1]:.4f}")
print(f"  最终 Policy Loss: {history['policy_loss'][-1]:.4f}")
print(f"  最终 Value Loss:  {history['value_loss'][-1]:.4f}")

## Part 9：生成质量对比

RLHF 训练的最终目的是改变模型的生成行为。让我们对比 RLHF 前后同一个 prompt 的生成结果。

我们加载 RLHF 前保存的权重来生成「训练前」的结果，然后用当前 Actor 生成「训练后」的结果。

In [ ]:
@torch.no_grad()
def generate_text(model, tokenizer, prompt, max_new_tokens=64, temperature=0.7):
    """用给定模型生成文本（贪心/采样）。"""
    model.eval()
    inputs = tokenizer(prompt, return_tensors='pt', padding=True, truncation=True,
                       max_length=32).to(device)
    
    generated = model.generate(
        input_ids=inputs['input_ids'],
        attention_mask=inputs['attention_mask'],
        max_new_tokens=max_new_tokens,
        temperature=temperature,
        do_sample=True,
        top_p=0.9,
        pad_token_id=tokenizer.pad_token_id,
    )
    
    full_text = tokenizer.decode(generated[0], skip_special_tokens=True)
    return full_text


# 用于对比的 prompts
test_prompts = [
    "Explain what machine learning is in simple terms.",
    "What are the benefits of regular exercise?",
    "How does the internet work?",
    "What is renewable energy?",
]

# 加载 RLHF 前的权重到一个临时模型
actor_before = GPT2LMHeadModel.from_pretrained('gpt2').to(device)
actor_before.load_state_dict(actor_before_state)
actor_before.eval()

print("=" * 70)
print("RLHF 前 vs 后 生成对比")
print("=" * 70)

for prompt in test_prompts:
    text_before = generate_text(actor_before, tokenizer, prompt)
    text_after = generate_text(actor, tokenizer, prompt)
    
    print(f"\n{'─' * 60}")
    print(f"Prompt: {prompt}")
    print(f"\n[BEFORE RLHF]:")
    print(f"  {text_before[:200]}")
    print(f"\n[AFTER RLHF]:")
    print(f"  {text_after[:200]}")

# 用 RM 对比打分
print(f"\n{'═' * 70}")
print("RM 打分对比")
print("═" * 70)

for prompt in test_prompts:
    text_before = generate_text(actor_before, tokenizer, prompt)
    text_after = generate_text(actor, tokenizer, prompt)
    
    enc_before = tokenizer(text_before, return_tensors='pt', max_length=128,
                           truncation=True, padding='max_length').to(device)
    enc_after = tokenizer(text_after, return_tensors='pt', max_length=128,
                          truncation=True, padding='max_length').to(device)
    
    score_before = reward_model(enc_before['input_ids'], enc_before['attention_mask']).item()
    score_after = reward_model(enc_after['input_ids'], enc_after['attention_mask']).item()
    
    delta = score_after - score_before
    arrow = "↑" if delta > 0 else "↓"
    print(f"  {prompt[:40]:40s} | Before: {score_before:+.3f} | After: {score_after:+.3f} | Δ: {delta:+.3f} {arrow}")

del actor_before

## Part 10：超参数分析

RLHF-PPO 中最关键的超参数是 **KL 系数 β**。我们用对比实验直观展示 β 对训练的影响：

| β 值 | 预期效果 |
|------|--------|
| β = 0.01 (小) | KL 约束弱 → Reward 上升快，但可能 Reward Hacking |
| β = 0.1 (中) | 平衡：Reward 稳步提升，KL 可控 |
| β = 0.5 (大) | KL 约束强 → Reward 提升慢，但更稳定 |

In [ ]:
def run_kl_experiment(kl_coef, n_iterations=30):
    """用指定 kl_coef 跑一轮 RLHF 训练（从头初始化）。"""
    exp_actor = GPT2LMHeadModel.from_pretrained('gpt2').to(device)
    exp_actor.config.pad_token_id = tokenizer.pad_token_id
    exp_critic = ValueHead('gpt2').to(device)
    
    hist = rlhf_train(
        actor=exp_actor, critic=exp_critic,
        reference=reference, reward_model=reward_model,
        tokenizer=tokenizer, prompts=PROMPTS,
        n_iterations=n_iterations, batch_size=4,
        max_new_tokens=48, lr_actor=1e-5, lr_critic=5e-5,
        kl_coef=kl_coef, clip_ratio=0.2, ppo_epochs=2,
        log_interval=100,  # 静默运行
    )
    
    del exp_actor, exp_critic
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return hist


print("运行 KL 系数对比实验...")
print("(每组约 30 iterations，请稍等)\n")

kl_experiments = {}
for beta in [0.01, 0.1, 0.5]:
    print(f"β = {beta} ...")
    kl_experiments[beta] = run_kl_experiment(beta, n_iterations=30)
    print(f"  → Final Reward: {kl_experiments[beta]['mean_reward'][-1]:+.3f}, "
          f"Final KL: {kl_experiments[beta]['mean_kl'][-1]:.4f}")

print("\n实验完成 ✓")

In [ ]:
# ========== KL 系数对比可视化 ==========
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
colors = {0.01: '#e74c3c', 0.1: '#3498db', 0.5: '#2ecc71'}

for beta, hist in kl_experiments.items():
    label = f'β={beta}'
    c = colors[beta]
    axes[0].plot(hist['mean_reward'], color=c, label=label, linewidth=2)
    axes[1].plot(hist['mean_kl'], color=c, label=label, linewidth=2)
    axes[2].plot(hist['response_len'], color=c, label=label, linewidth=2)

axes[0].set_title('Mean Reward vs β')
axes[0].set_xlabel('Iteration')
axes[0].set_ylabel('Reward')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].set_title('Mean KL vs β')
axes[1].set_xlabel('Iteration')
axes[1].set_ylabel('KL')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

axes[2].set_title('Response Length vs β')
axes[2].set_xlabel('Iteration')
axes[2].set_ylabel('Tokens')
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.suptitle('KL Coefficient (β) Ablation Study', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print("\n分析:")
print("  β 小 → Reward 上升快但 KL 可能爆炸 (Reward Hacking 风险)")
print("  β 大 → KL 被强力约束，Reward 提升慢但更稳定")
print("  β 适中 → 平衡探索与约束，是实践中的最佳选择")

## Part 11：面试要点 + 总结

### 11.1 核心代码速查

**面试要求**：能闭眼写出 RLHF-PPO 一轮 iteration 的完整流程。

```python
# Phase 1: Rollout
response = actor.generate(prompt)
old_log_probs = actor.log_prob(prompt, response)
old_values = critic.value(prompt, response)

# Phase 2: Scoring
rm_score = reward_model(prompt, response)
ref_log_probs = reference.log_prob(prompt, response)
kl = old_log_probs - ref_log_probs
rewards = -beta * kl
rewards[-1] += rm_score

# Phase 3: GAE
for t in reversed(range(T)):
    delta = rewards[t] + gamma * V[t+1] - V[t]
    A[t] = delta + gamma * lam * A[t+1]
returns = A + old_values

# Phase 4: PPO Update
new_log_probs = actor.log_prob(prompt, response)  # 重算，需要梯度
ratio = exp(new_log_probs - old_log_probs)
L_policy = -min(ratio * A, clip(ratio, 1±ε) * A)
L_value = (new_values - returns) ** 2
loss = L_policy + c_v * L_value
loss.backward()
```

### 11.2 关键设计决策

| 决策 | 本实现的选择 | 理由 |
|------|------------|------|
| KL 实现方式 | Reward Shaping | 缓解稀疏奖励，每个 token 都有信号 |
| Actor-Critic | 独立 backbone | 避免梯度冲突，易调试 |
| γ | 1.0 | Response 较短，不需折扣 |
| Advantage 标准化 | 每个 batch | 稳定梯度尺度 |

### 11.3 常见面试追问

1. **为什么 rollout 用 no_grad，update 用 grad？**  
   → rollout 是采集数据（on-policy），不需要更新；update 才是反向传播训练。

2. **为什么 PPO 用 old_log_probs 而不是直接用梯度？**  
   → PPO 是 off-policy correction：用旧策略采样的数据更新新策略，ratio 做重要性权重。

3. **KL 爆炸了怎么办？**  
   → 增大 β，或使用 adaptive KL（根据 mean_kl 动态调整 β）。

4. **Reward 一直涨但生成质量下降？**  
   → 典型的 Reward Hacking，说明 RM 有偏差。检查 KL 是否同步增长。

---

### 11.4 总结

本日我们从零实现了完整的 RLHF-PPO 训练循环：

1. **四模型构建**：Actor / Critic / Reference / Reward Model
2. **Rollout**：Actor 自回归生成，收集 log_probs 和 values
3. **Reward 计算**：RM 打分 + per-token KL penalty
4. **GAE**：反向递推计算 Advantage 和 Returns
5. **PPO Update**：Clipped policy loss + Value loss，梯度更新 Actor 和 Critic
6. **训练监控**：Reward / KL / Loss 曲线
7. **生成对比**：RLHF 前后的生成质量变化
8. **超参数分析**：β 对 Reward-KL 权衡的影响

### 下一步

Day 7 将对本周的 RLHF 实践进行深入分析与复盘——讨论 Reward Hacking 的根源与对策、RLHF 的工程挑战与局限性，以及 DPO（Direct Preference Optimization）如何绕过 RM 和 PPO，为 Week 11 做铺垫。